# CEG-WM Content V2 — Runtime Asset Contract V3 formal initial user-only GPU run

This Colab notebook executes the personally approved frozen **initial invocation only** for `content-adaptive-v2-e3fe3fd32ca2-805bc21e173a`. It checks out and verifies exact `4d8b0df5bf7840d242115669f1d3115cdf6810cc`, validates the fixed protocol and roster, mounts the user's Drive, reads secrets from Colab Secrets, and invokes the existing formal runner exactly once.

Before starting, create Colab Secrets named `CEG_WM_ROOT_KEY` and `HF_TOKEN`. Run Cells 1-3 once and in order. Cell 4 only downloads existing artifacts and never runs or resumes the experiment. Do not retry or resume after any failure, interruption, OOM, Drive issue, or runtime loss.

## 1. Fresh checkout and frozen identity proof

This cell must run before dependency installation, secret access, Drive mounting, or GPU/model work. Any existing source directory or identity mismatch stops the handoff.

In [ ]:
import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "Content-V2-Evidence"
EXACT = "4d8b0df5bf7840d242115669f1d3115cdf6810cc"
PUBLIC_KEY_DIGEST = "805bc21e173a83898f3b7034d75e6ed02f65894a6885377d9659ee3091b4dd77"
RUN_ID = "content-adaptive-v2-e3fe3fd32ca2-805bc21e173a"

repo = pathlib.Path("/content/cegwm-stage-a-content-adaptive-dual-branch-v2-source")
local_work_root = pathlib.Path("/content/cegwm-stage-a-content-adaptive-dual-branch-v2-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/stage_a_content_adaptive_dual_branch_v2_clean")

_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "ValueError",
}

def stop(stage, error_class="RuntimeError"):
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    print("CEGWM_FORMAL_HANDOFF_FAILURE " + json.dumps(
        {"stage": stage, "error_class": error_class},
        sort_keys=True, separators=(",", ":"),
    ))
    raise SystemExit

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

try:
    if repo.exists():
        raise FileExistsError
    clone = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError
    git("checkout", "--detach", EXACT)
    if (
        git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain") != ""
    ):
        raise RuntimeError
except BaseException as error:
    stop("source_checkout_identity_validation", type(error).__name__)


## 2. Install the checked-out project

This installs only the requirements declared by the frozen checkout, then proves that the source identity and clean state have not changed. Do not retry or change versions if installation fails.

In [ ]:
try:
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", str(repo)],
        capture_output=True, text=True,
    )
    if install.returncode != 0:
        raise RuntimeError
    if (
        git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain") != ""
    ):
        raise RuntimeError
except BaseException as error:
    stop("dependency_install_and_source_recheck", type(error).__name__)


## 3. Mount Drive, validate Secrets, and invoke the formal runner once

Create Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN` before running this cell. The public digest of the normalized root key must match the frozen protocol identity. Any existing local or Drive run destination stops the initial invocation. This cell invokes the formal runner exactly once and intentionally suppresses raw stderr.

In [ ]:
import os
from google.colab import drive, userdata
from cegwm.shared.keys import normalize_detection_key, public_key_digest

try:
    drive.mount("/content/drive")
    bound_local_run = local_work_root / RUN_ID
    bound_drive_run = artifact_sink / RUN_ID

    # INITIAL INVOCATION ONLY: any existing bound destination stops the run.
    if bound_local_run.exists() or bound_drive_run.exists():
        raise FileExistsError
    artifact_sink.mkdir(parents=True, exist_ok=True)

    root_key = userdata.get("CEG_WM_ROOT_KEY")
    hf_token = userdata.get("HF_TOKEN")
    if not isinstance(root_key, str) or not root_key.strip():
        raise RuntimeError
    if not isinstance(hf_token, str) or not hf_token.strip():
        raise RuntimeError

    normalized_key = normalize_detection_key(root_key)
    if public_key_digest(normalized_key) != PUBLIC_KEY_DIGEST:
        raise RuntimeError
    normalized_key = b""

    if (
        git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain") != ""
        or bound_local_run.exists()
        or bound_drive_run.exists()
    ):
        raise RuntimeError
except BaseException as error:
    root_key = hf_token = ""
    stop("initial_destination_and_secret_identity_validation", type(error).__name__)

runner_env = os.environ.copy()
runner_env["CEG_WM_ROOT_KEY"] = root_key
runner_env["HF_TOKEN"] = hf_token
runner_exception_class = None
try:
    completed = subprocess.run(
        [
            sys.executable,
            "-m", "experiments.run_content_adaptive_dual_branch_v2_clean",
            "--repo-root", str(repo),
            "--expected-exact", EXACT,
            "--local-work-root", str(local_work_root),
            "--artifact-sink", str(artifact_sink),
        ],
        cwd=repo,
        env=runner_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
    )
    runner_rc = completed.returncode
    runner_stdout = completed.stdout
except BaseException as error:
    runner_rc = None
    runner_stdout = ""
    runner_exception_class = type(error).__name__
finally:
    runner_env.pop("CEG_WM_ROOT_KEY", None)
    runner_env.pop("HF_TOKEN", None)
    root_key = hf_token = ""
    del runner_env


## 4. Download existing terminal or checkpoint artifacts

Run this cell once after normal completion or after reconnecting following an interruption. It is self-contained, mounts Drive if necessary, and never invokes or resumes the runner. A terminal run returns one ZIP/SHA256 pair. An interrupted run returns every complete checkpoint ZIP/SHA256 pair.

In [ ]:
import json
import pathlib
from google.colab import drive, files

RUN_ID = "content-adaptive-v2-e3fe3fd32ca2-805bc21e173a"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/stage_a_content_adaptive_dual_branch_v2_clean")

def artifact_stop(stage):
    print("CEGWM_FORMAL_HANDOFF_FAILURE " + json.dumps(
        {"stage": stage, "error_class": "RuntimeError"},
        sort_keys=True, separators=(",", ":"),
    ))
    raise SystemExit

if not pathlib.Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

run_dir = artifact_sink / RUN_ID
terminal_zip = run_dir / f"{RUN_ID}.zip"
terminal_sha = run_dir / f"{RUN_ID}.zip.sha256"

if terminal_zip.is_file() and terminal_sha.is_file():
    files.download(str(terminal_zip))
    files.download(str(terminal_sha))
elif terminal_zip.exists() or terminal_sha.exists():
    artifact_stop("artifact_pair_validation")
else:
    checkpoint_zips = sorted(run_dir.glob(f"{RUN_ID}.checkpoint-*.zip"))
    checkpoint_shas = sorted(run_dir.glob(f"{RUN_ID}.checkpoint-*.zip.sha256"))
    expected_shas = {pathlib.Path(str(path) + ".sha256") for path in checkpoint_zips}
    if checkpoint_zips and set(checkpoint_shas) == expected_shas:
        for path in checkpoint_zips:
            files.download(str(path))
            files.download(str(path) + ".sha256")
    else:
        artifact_stop("approved_artifact_discovery")


## Return and stop boundary

- **Terminal completion:** return exactly the downloaded terminal ZIP and `.sha256` sidecar.
- **Interruption:** run Cell 4 once and return every downloaded checkpoint ZIP/`.sha256` pair. Do not relaunch.
- **Handoff failure:** return only the single `CEGWM_FORMAL_HANDOFF_FAILURE` line. Never expose raw stdout, stderr, traceback, Secrets, or private state.
- Never retry after RC0/RC1/RC2, exception, interruption, OOM, dependency/model/backend error, missing artifact, Drive issue, or Colab loss.
- Resume requires a new user-personal approval and independent checkpoint-chain inspection.
- Do not change the exact, roots, model, backend, dtype, prompts, seeds, roster, budget, probes, Gates, ties, threshold, key/PRG, wrong keys, or primary null.
- Do not interpret results or make a scientific or promotion decision from this notebook.